In [5]:
import torch
import torch.nn as nn

from torchvision.models import mobilenet_v2,MobileNet_V2_Weights

In [7]:
mobilenet = mobilenet_v2(weights=None)
features = mobilenet.features

In [10]:
dummy_tensor=torch.randn((5,3,96,96))
features(dummy_tensor).shape

torch.Size([5, 1280, 3, 3])

# Coarse Feature Extractor

In [22]:
class CoarseFeatureExtractor(nn.Module):
    def __init__(self):
        super().__init__()
        self.features=mobilenet_v2(weights=MobileNet_V2_Weights.IMAGENET1K_V2).features

    def forward(self,x:torch.Tensor):
        B,T,C,H,W=x.shape
        x=x.view(B*T,C,H,W)
        feat=self.features(x)
        _,C2,H2,W2=feat.shape
        feat=feat.view(B,T,C2,H2,W2)
        return feat

# TSD block

`Temporal Sequence Distillation block`

In [23]:
class TSDBlock(nn.Module):
  def __init__(self,T,Ts,in_channels=1280) -> None:
    super().__init__()

    self.T=T
    
    self.Ts=Ts

    self.conv_alpha = nn.Conv3d(in_channels,in_channels,kernel_size=1)

    self.conv_beta = nn.Conv3d(T,Ts,kernel_size=1)

    self.conv_gamma = nn.Conv3d(in_channels,in_channels,kernel_size=1)

  def forward(self,x:torch.Tensor):
    B,T,_,_,_ = x.shape

    x=x.transpose(1,2)       # (B, C, T, H, W)

    O=self.conv_alpha(x)     # (B, C, T, H, W)

    O=O.transpose(1,2)       # (B, T, C, H, W)

    O=self.conv_beta(O)      # (B, Ts, C, H, W)

    O=O.flatten(2)           # (B, Ts, C*H*W)

    O=O.permute(0,2,1)       # (B, C*H*W, Ts)

    G=self.conv_gamma(x)
    G=G.transpose(1,2)
    G=G.reshape(B,T,-1)
    P=torch.bmm(G,O)
    P=torch.softmax(P,dim=1)
    return P

# Distillation frames

In [21]:
def distill_frames(X,P):
    B,T,C,H,W=X.shape
    Ts=P.shape[-1]
    X_flat=X.view(B,T,-1)
    Y=torch.matmul(X_flat.transpose(1,2),P)
    Y=Y.transpose(1,2)
    Y=Y.view(B,Ts,C,H,W)
    return Y

In [26]:
coarse_feature_network=CoarseFeatureExtractor()
tsd_block=TSDBlock(T=150,Ts=20)


dummy_tensor=torch.randn((1,150,3,96,96))
features=coarse_feature_network(dummy_tensor)
P_matrix=tsd_block(features)
final_frames=distill_frames(dummy_tensor,P_matrix)

In [28]:
final_frames.shape, P_matrix.shape

(torch.Size([1, 20, 3, 96, 96]), torch.Size([1, 150, 20]))

In [31]:
P_matrix[0][0]

tensor([6.6247e-36, 2.5218e-18, 1.3111e-13, 7.7559e-11, 1.9991e-24, 3.7518e-24,
        1.4689e-12, 1.5043e-21, 1.4674e-19, 1.6218e-23, 3.0589e-15, 7.2293e-22,
        1.2475e-34, 1.2044e-17, 6.2394e-20, 4.0078e-21, 7.3019e-14, 3.1528e-32,
        1.0063e-22, 2.0420e-19], grad_fn=<SelectBackward0>)

# TSD Network

In [18]:
# class TSDNetwork(nn.Module):

#     def __init__(self, Ts=20):

#         super().__init__()

#         self.feature_extractor = CoarseFeatureExtractor()

#         self.tsd = TSDBlock(channels=1280, Ts=Ts)

#         self.main_net = MainNetwork()

#     def forward(self, frames):

#         """
#         frames: (B, T, 3, H, W)
#         """

#         features = self.feature_extractor(frames)

#         P = self.tsd(features)

#         distilled = distill_frames(frames, P)

#         out = self.main_net(distilled)

#         return out

# MultiHead KNN Attention

In [13]:
class MultiHeadKNNAttention(nn.Module):
    def __init__(self, dim, heads,k=16):
        super().__init__()
        self.heads = heads
        self.k = k
        self.head_dim = dim//heads
        self.scale = self.head_dim**-0.5

        self.to_qkv=nn.Linear(dim,dim*3)
        self.to_out=nn.Linear(dim,dim)

    def forward(self,x:torch.tensor):

        B,N,D = x.shape

        qkv = self.to_qkv(x)

        qkv = qkv.view(B,N,3,self.heads,self.head_dim)

        print(f"Shape qkv: {qkv.shape}")

        q,k,v = qkv.unbind(dim=2)

        print(f"Shape q: {q.shape}")

        q=q.transpose(1,2) # B,heads,N,head_dim
        k=k.transpose(1,2)
        v=v.transpose(1,2)

        scores = torch.matmul(q,k.transpose(-2,-1))*self.scale

        print(scores)
        print(scores.shape)

        topk_vals,topk_idx=torch.topk(scores,self.k,dim=-1)

        attn = torch.softmax(topk_vals,dim=-1)

        v_exp = v.unsqueeze(dim=-3).expand(-1,-1,N,-1,-1)
        topk_v = torch.gather(
            v_exp,
            -2,
            topk_idx.unsqueeze(dim=-1).expand(-1,-1,-1,-1,self.head_dim)
        )

        out = torch.sum(attn.unsqueeze(-1) * topk_v,dim=-2)
        
        out = out.transpose(1,2).reshape(B,N,D)

        return self.to_out(out)

In [14]:
knn_attention=MultiHeadKNNAttention(dim=6,heads=2,k=4)

In [15]:
dummy_tensor=torch.randn((3,6,6))
knn_attention(dummy_tensor).shape

Shape qkv: torch.Size([3, 6, 3, 2, 3])
Shape q: torch.Size([3, 6, 2, 3])
tensor([[[[-9.8331e-02,  3.3217e-01,  1.3603e-01,  3.2818e-01, -1.7157e-01,
            3.6558e-01],
          [-4.4425e-02,  2.8439e-01, -1.8858e-01,  4.3482e-02,  3.7146e-01,
           -2.7673e-01],
          [ 1.3190e-02,  4.1538e-01, -7.0376e-02,  4.6197e-01, -2.5554e-01,
            4.1959e-01],
          [ 2.1580e-01, -3.3202e-03, -4.7521e-01, -2.3459e-02,  1.5034e-01,
           -2.9570e-01],
          [ 4.6783e-02,  3.0903e-01,  1.2256e-01,  6.1608e-01, -7.4378e-01,
            8.9405e-01],
          [-1.9161e-01,  8.9646e-01, -2.1408e-01,  4.2865e-01,  5.3313e-01,
           -1.4511e-01]],

         [[ 8.3365e-02, -3.7793e-01,  3.4445e-01,  6.9571e-01, -6.3810e-01,
            8.0907e-01],
          [-3.4108e-02,  3.1842e-01, -1.2803e-01, -9.2530e-01,  9.6102e-01,
           -1.1235e+00],
          [ 4.0457e-02, -3.0549e-01, -1.6832e-01,  1.5194e+00,  2.1125e-02,
            3.7053e-01],
          [-1.07

torch.Size([3, 6, 6])

In [6]:
knn_attention=MultiHeadKNNAttention(dim=1024,heads=8,k=10)
dummy_tensor=torch.randn((1,20,1024))
knn_attention(dummy_tensor)

torch.Size([1, 8, 20, 128])
